# NeuroBridge-S4 Graph Learning — Phase 7: Explainable Within-Subject Trajectory Attribution

> **Phase 6 asked:** *How did the subject's biological adaptation graph change from personal baseline over time?*
>
> **Phase 7 asks:** *Which biological domains, subgraphs, HRP hazard contexts, and recovery components explain that change?*

Phase 7 is a transparent, reviewer-facing **attribution layer**:

```text
baseline-relative graph change
  -> node contributors
  -> subgraph contributors
  -> graph-metric contributors
  -> hazard-context contributors
  -> recovery contributors
  -> plain-language trajectory explanation
```


## 1. Plain-language purpose

Phase 6 measured *how* each subject's biological adaptation graph changed relative to personal baseline. Phase 7 explains *which* domains, subgraphs, graph metrics, hazard contexts, and recovery components contributed most to those changes.

Attribution is **transparent arithmetic**, not a black-box model. The core formula is:

```text
domain contribution share = absolute domain delta
                            / total absolute domain delta for that subject-timepoint
```

A reviewer can verify every number by hand.

## 2. Why attribution matters

For small-N astronaut monitoring it is not enough to know that a graph changed. Reviewers need to know **what drove the change** and whether it appears localized, distributed, persistent, or recovery-associated.

Attribution turns a single delta number into an evidence trail: which biological systems moved, how those systems cluster into subgraphs, which HRP hazard contexts they align with, and whether the shift recovered toward baseline.

## 3. Scientific guardrails

- This is **not** diagnosis.
- This is **not** treatment guidance.
- This is **not** causal attribution.
- HRP hazard-context attribution is **hazard-context alignment, not exposure attribution**.
- If example data from Phase 6 are used, they are **schema demonstration only** and are not scientific evidence.
- Outputs are **monitoring-relevant research interpretations** and **candidates for expert review**.

## Setup

In [1]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

from neurobridge_graph.trajectory_attribution import (
    load_phase6_delta_tables,
    compute_node_attribution,
    compute_graph_metric_attribution,
    compute_subgraph_attribution_from_node_deltas,
    compute_hazard_context_attribution,
    compute_recovery_attribution,
    build_phase7_attribution_summary,
    DEFAULT_SUBGRAPH_TEMPLATES,
)
from neurobridge_graph.explanation_generator import (
    generate_subject_timepoint_explanation,
    generate_phase7_report,
)
from neurobridge_graph.attribution_visualization import (
    plot_node_attribution_bar,
    plot_subgraph_attribution_heatmap,
    plot_hazard_context_attribution_heatmap,
    plot_recovery_attribution_summary,
    plot_subject_explanation_panel,
)

RESULTS = repo_root / 'results'
TABLES = RESULTS / 'tables'
FIGURES = RESULTS / 'figures'
REPORTS = RESULTS / 'reports'
CARDS = REPORTS / 'explanation_cards'
for d in [TABLES, FIGURES, REPORTS, CARDS]:
    d.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print('Setup complete.')


Setup complete.


## 5. Load Phase 6 delta tables

Phase 7 consumes Phase 6 outputs. Core required tables are `longitudinal_node_deltas.csv` and `longitudinal_graph_deltas.csv`. The hazard, recovery, and trajectory-summary tables are optional — if they are missing, the corresponding attribution layers are simply reported as unavailable.

In [2]:
REQUIRED = {
    'node_deltas': 'longitudinal_node_deltas.csv',
    'graph_deltas': 'longitudinal_graph_deltas.csv',
}
OPTIONAL = {
    'hazard_deltas': 'longitudinal_hazard_deltas.csv',
    'recovery_metrics': 'recovery_metrics.csv',
    'trajectory_summary': 'longitudinal_trajectory_summary.csv',
}

core_missing = [f for f in REQUIRED.values() if not (TABLES / f).exists()]
if core_missing:
    raise SystemExit(
        'Phase 7 requires Phase 6 longitudinal delta outputs. '
        'Please run the Phase 6 notebook first. Missing: ' + str(core_missing)
    )

tables = load_phase6_delta_tables(RESULTS)

node_deltas = tables['node_deltas']
graph_deltas = tables['graph_deltas']
hazard_deltas = tables.get('hazard_deltas')
recovery_metrics = tables.get('recovery_metrics')
trajectory_summary = tables.get('trajectory_summary')

# Detect schema-demonstration provenance from the Phase 6 data_type column.
provenance = 'unknown'
if 'data_type' in node_deltas.columns and not node_deltas.empty:
    provenance = str(node_deltas['data_type'].iloc[0])
is_demo = 'schema_demonstration' in provenance
provenance_note = (
    'Data provenance: schema demonstration only — not scientific evidence.'
    if is_demo else f'Data provenance: {provenance}.'
)
print(provenance_note)


Data provenance: schema demonstration only — not scientific evidence.


Build and save the input readiness check.

In [3]:
readiness_rows = []
for key, fname in {**REQUIRED, **OPTIONAL}.items():
    fpath = TABLES / fname
    required_or_optional = 'required' if fname in REQUIRED.values() else 'optional'
    if fpath.exists():
        df = tables.get(key)
        rows = 0 if df is None else len(df)
        cols = 0 if df is None else df.shape[1]
        status, notes = 'found', 'loaded'
    else:
        rows, cols = 0, 0
        status = 'missing'
        notes = ('blocks Phase 7' if required_or_optional == 'required'
                 else 'attribution layer unavailable')
    readiness_rows.append({
        'file': fname, 'status': status, 'rows': rows, 'columns': cols,
        'required_or_optional': required_or_optional, 'notes': notes,
    })

readiness = pd.DataFrame(readiness_rows)
readiness.to_csv(TABLES / 'phase7_input_readiness_check.csv', index=False)
readiness


,file,status,rows,columns,required_or_optional,notes
0,longitudinal_node_deltas.csv,found,60,12,required,loaded
1,longitudinal_graph_deltas.csv,found,70,10,required,loaded
2,longitudinal_hazard_deltas.csv,found,50,9,optional,loaded
3,recovery_metrics.csv,found,8,11,optional,loaded
4,longitudinal_trajectory_summary.csv,found,10,15,optional,loaded


## 6. Compute node-level trajectory attribution

Node contribution share describes which biological domains account for the largest share of the baseline-relative graph shift. Shares sum to 1 within each subject-timepoint that shows any change, and to 0 at baseline (no shift to attribute).

In [4]:
node_attr = compute_node_attribution(node_deltas)
node_attr.to_csv(TABLES / 'trajectory_node_attribution.csv', index=False)
print(f'Node attribution rows: {len(node_attr)}')

# Top contributors at each non-baseline subject-timepoint.
top_nodes = (
    node_attr[node_attr['contribution_share'] > 0]
    .sort_values(['subject_id', 'timepoint', 'contribution_share'],
                 ascending=[True, True, False])
    .groupby(['subject_id', 'timepoint']).head(3)
)
top_nodes[['subject_id', 'timepoint', 'mission_phase', 'domain',
           'delta_activation', 'contribution_share', 'attribution_rank']]


Node attribution rows: 60


,subject_id,timepoint,mission_phase,domain,delta_activation,contribution_share,attribution_rank
6,Demo_Crew_01,T1_pre,pre_mission,Cardiovascular regulation,0.03,0.21429,1
7,Demo_Crew_01,T1_pre,pre_mission,Inflammation / immune-adjacent,0.03,0.21429,2
8,Demo_Crew_01,T1_pre,pre_mission,Body composition / physical status,0.02,0.14286,3
12,Demo_Crew_01,T2_inflight,inflight,Body composition / physical status,0.40,0.26144,1
13,Demo_Crew_01,T2_inflight,inflight,Inflammation / immune-adjacent,0.40,0.26144,2
14,Demo_Crew_01,T2_inflight,inflight,Cardiovascular regulation,0.25,0.16340,3
18,Demo_Crew_01,T3_post,postflight,Body composition / physical status,0.55,0.23605,1
19,Demo_Crew_01,T3_post,postflight,Inflammation / immune-adjacent,0.55,0.23605,2
20,Demo_Crew_01,T3_post,postflight,Cardiovascular regulation,0.40,0.17167,3
24,Demo_Crew_01,T4_recovery,recovery,Inflammation / immune-adjacent,0.13,0.24074,1


## 7. Compute graph-metric attribution

Graph-metric attribution shows whether the trajectory is driven by broad activation, peak activation, active-domain count, connectivity, or co-activation changes.

In [5]:
graph_attr = compute_graph_metric_attribution(graph_deltas)
graph_attr.to_csv(TABLES / 'trajectory_graph_metric_attribution.csv', index=False)
print(f'Graph-metric attribution rows: {len(graph_attr)}')
graph_attr[graph_attr['contribution_share'] > 0][
    ['subject_id', 'timepoint', 'metric', 'delta_value',
     'contribution_share', 'attribution_rank']
].head(12)


Graph-metric attribution rows: 70


,subject_id,timepoint,metric,delta_value,contribution_share,attribution_rank
7,Demo_Crew_01,T1_pre,total_node_activation,0.14000,0.76361,1
8,Demo_Crew_01,T1_pre,mean_node_activation,0.02334,0.12730,2
9,Demo_Crew_01,T1_pre,max_node_activation,0.02000,0.10909,3
14,Demo_Crew_01,T2_inflight,coactivation_edge_count,6.00000,0.41904,1
15,Demo_Crew_01,T2_inflight,n_active_domains,5.00000,0.34920,2
16,Demo_Crew_01,T2_inflight,total_node_activation,1.53000,0.10686,3
17,Demo_Crew_01,T2_inflight,active_domain_fraction,0.83333,0.05820,4
18,Demo_Crew_01,T2_inflight,graph_density,0.40000,0.02794,5
19,Demo_Crew_01,T2_inflight,max_node_activation,0.30000,0.02095,6
20,Demo_Crew_01,T2_inflight,mean_node_activation,0.25500,0.01781,7


## 8. Compute subgraph attribution

Subgraph attribution asks whether change is concentrated in biologically meaningful clusters such as cardiometabolic, immune-metabolic, or sleep-autonomic-recovery patterns. Domains that are absent from the dataset are reported with zero available domains rather than silently dropped.

Default subgraph templates:

In [6]:
for name, domains in DEFAULT_SUBGRAPH_TEMPLATES.items():
    print(f'- {name}: ' + ', '.join(domains))


- cardiometabolic: cardiovascular regulation, metabolic regulation, body composition / physical status
- immune_metabolic: inflammation / immune-adjacent status, metabolic regulation, recovery-related markers
- hematologic_cardiovascular: hematologic / oxygen-carrying capacity, cardiovascular regulation, recovery-related markers
- sleep_autonomic_recovery: sleep / circadian regulation, autonomic regulation, recovery capacity
- cognitive_emotional_recovery: cognitive load, emotional regulation, recovery capacity, recovery-related markers


In [7]:
subgraph_attr = compute_subgraph_attribution_from_node_deltas(node_attr)
subgraph_attr.to_csv(TABLES / 'trajectory_subgraph_attribution.csv', index=False)
print(f'Subgraph attribution rows: {len(subgraph_attr)}')
subgraph_attr[subgraph_attr['n_available_domains'] > 0][
    ['subject_id', 'timepoint', 'subgraph_name', 'n_available_domains',
     'total_contribution_share', 'dominant_domain']
].head(12)


Subgraph attribution rows: 50


,subject_id,timepoint,subgraph_name,n_available_domains,total_contribution_share,dominant_domain
0,Demo_Crew_01,T0_baseline,cardiometabolic,3,0.00000,Body composition / physical status
1,Demo_Crew_01,T0_baseline,immune_metabolic,3,0.00000,Inflammation / immune-adjacent
2,Demo_Crew_01,T0_baseline,hematologic_cardiovascular,3,0.00000,Cardiovascular regulation
4,Demo_Crew_01,T0_baseline,cognitive_emotional_recovery,1,0.00000,Recovery-related markers
5,Demo_Crew_01,T1_pre,cardiometabolic,3,0.50001,Cardiovascular regulation
6,Demo_Crew_01,T1_pre,immune_metabolic,3,0.50001,Inflammation / immune-adjacent
7,Demo_Crew_01,T1_pre,hematologic_cardiovascular,3,0.50001,Cardiovascular regulation
9,Demo_Crew_01,T1_pre,cognitive_emotional_recovery,1,0.14286,Recovery-related markers
10,Demo_Crew_01,T2_inflight,cardiometabolic,3,0.52288,Body composition / physical status
11,Demo_Crew_01,T2_inflight,immune_metabolic,3,0.44445,Inflammation / immune-adjacent


## 9. Compute hazard-context attribution

Hazard-context attribution identifies which HRP hazard categories the graph shift maps onto most strongly. **It does not indicate actual exposure or causal effect** — it is hazard-context alignment only.

In [8]:
if hazard_deltas is not None:
    hazard_attr = compute_hazard_context_attribution(hazard_deltas)
    hazard_attr.to_csv(TABLES / 'trajectory_hazard_attribution.csv', index=False)
    print(f'Hazard-context attribution rows: {len(hazard_attr)}')
    display(hazard_attr[hazard_attr['contribution_share'] > 0][
        ['subject_id', 'timepoint', 'hazard', 'delta_hazard_relevance',
         'contribution_share', 'attribution_rank']
    ].head(12))
else:
    hazard_attr = None
    print('Hazard delta table not available — hazard-context attribution skipped.')


Hazard-context attribution rows: 50


,subject_id,timepoint,hazard,delta_hazard_relevance,contribution_share,attribution_rank
5,Demo_Crew_01,T1_pre,isolation_and_confinement,0.03000,0.22862,1
6,Demo_Crew_01,T1_pre,distance_from_earth,0.03000,0.22862,2
7,Demo_Crew_01,T1_pre,space_radiation,0.02500,0.19052,3
8,Demo_Crew_01,T1_pre,hostile_closed_environments,0.02400,0.18290,4
9,Demo_Crew_01,T1_pre,gravity_fields,0.02222,0.16933,5
10,Demo_Crew_01,T2_inflight,isolation_and_confinement,0.40000,0.28824,1
11,Demo_Crew_01,T2_inflight,space_radiation,0.25500,0.18375,2
12,Demo_Crew_01,T2_inflight,distance_from_earth,0.25000,0.18015,3
13,Demo_Crew_01,T2_inflight,hostile_closed_environments,0.24300,0.17511,4
14,Demo_Crew_01,T2_inflight,gravity_fields,0.23972,0.17274,5


## 10. Compute recovery attribution

Recovery attribution interprets which metrics recovered toward baseline, remained shifted, or overshot. Categories:

- **returned_near_baseline** — recovered close to personal baseline;
- **partial_recovery** — moved back partway;
- **persistent_shift** — remained shifted from baseline;
- **overshoot_or_reversal** — crossed past baseline to the opposite side;
- **insufficient_data** — not enough information to assess recovery.

In [9]:
if recovery_metrics is not None:
    recovery_attr = compute_recovery_attribution(recovery_metrics)
    recovery_attr.to_csv(TABLES / 'recovery_attribution.csv', index=False)
    print(f'Recovery attribution rows: {len(recovery_attr)}')
    print('Category counts:', recovery_attr['recovery_category'].value_counts().to_dict())
    display(recovery_attr[['subject_id', 'metric', 'recovery_fraction',
                           'recovery_category']])
else:
    recovery_attr = None
    print('Recovery metrics table not available — recovery attribution skipped.')


Recovery attribution rows: 8
Category counts: {'returned_near_baseline': 8}


,subject_id,metric,recovery_fraction,recovery_category
0,Demo_Crew_01,mean_node_activation,0.76824,returned_near_baseline
1,Demo_Crew_01,max_node_activation,0.82222,returned_near_baseline
2,Demo_Crew_01,total_node_activation,0.76824,returned_near_baseline
3,Demo_Crew_01,n_active_domains,1.00000,returned_near_baseline
4,Demo_Crew_02,mean_node_activation,0.80500,returned_near_baseline
5,Demo_Crew_02,max_node_activation,0.83333,returned_near_baseline
6,Demo_Crew_02,total_node_activation,0.80500,returned_near_baseline
7,Demo_Crew_02,n_active_domains,1.00000,returned_near_baseline


## 11. Build attribution summary

One row per subject-timepoint, naming the top contributor in each layer plus a plain-language interpretation.

In [10]:
attribution_summary = build_phase7_attribution_summary(
    node_attr, graph_attr, subgraph_attr, hazard_attr, recovery_attr
)
attribution_summary.to_csv(TABLES / 'phase7_attribution_summary.csv', index=False)
print(f'Summary rows: {len(attribution_summary)}')
attribution_summary


Summary rows: 10


,subject_id,timepoint,mission_phase,top_domain_contributor,top_domain_contribution_share,top_graph_metric_contributor,top_subgraph_contributor,top_hazard_context_contributor,recovery_summary,interpretation
0,Demo_Crew_01,T0_baseline,baseline,Body composition / physical status,0.00000,mean_node_activation,cardiometabolic,n/a,returned_near_baseline:4,"At T0_baseline (baseline), the graph matches p..."
1,Demo_Crew_01,T1_pre,pre_mission,Cardiovascular regulation,0.21429,total_node_activation,cardiometabolic,isolation_and_confinement,returned_near_baseline:4,"At T1_pre (pre_mission), the baseline-relative..."
2,Demo_Crew_01,T2_inflight,inflight,Body composition / physical status,0.26144,coactivation_edge_count,cardiometabolic,isolation_and_confinement,returned_near_baseline:4,"At T2_inflight (inflight), the baseline-relati..."
3,Demo_Crew_01,T3_post,postflight,Body composition / physical status,0.23605,coactivation_edge_count,cardiometabolic,isolation_and_confinement,returned_near_baseline:4,"At T3_post (postflight), the baseline-relative..."
4,Demo_Crew_01,T4_recovery,recovery,Inflammation / immune-adjacent,0.24074,total_node_activation,cardiometabolic,isolation_and_confinement,returned_near_baseline:4,"At T4_recovery (recovery), the baseline-relati..."
5,Demo_Crew_02,T0_baseline,baseline,Body composition / physical status,0.00000,mean_node_activation,cardiometabolic,n/a,returned_near_baseline:4,"At T0_baseline (baseline), the graph matches p..."
6,Demo_Crew_02,T1_pre,pre_mission,Metabolic regulation,0.23077,total_node_activation,cardiometabolic,hostile_closed_environments,returned_near_baseline:4,"At T1_pre (pre_mission), the baseline-relative..."
7,Demo_Crew_02,T2_inflight,inflight,Metabolic regulation,0.23179,n_active_domains,cardiometabolic,isolation_and_confinement,returned_near_baseline:4,"At T2_inflight (inflight), the baseline-relati..."
8,Demo_Crew_02,T3_post,postflight,Metabolic regulation,0.22500,n_active_domains,cardiometabolic,isolation_and_confinement,returned_near_baseline:4,"At T3_post (postflight), the baseline-relative..."
9,Demo_Crew_02,T4_recovery,recovery,Metabolic regulation,0.25641,total_node_activation,cardiometabolic,isolation_and_confinement,returned_near_baseline:4,"At T4_recovery (recovery), the baseline-relati..."


## 12. Visualize attribution

Figures are intentionally plain. Each carries a guardrail caption: these visualize transparent attribution of within-subject graph change — not diagnosis, causal proof, or exposure measurement.

In [11]:
fig_paths = {}
fig_paths['node'] = plot_node_attribution_bar(
    node_attr, FIGURES / 'phase7_node_attribution_bar.png', top_n=8)
fig_paths['subgraph'] = plot_subgraph_attribution_heatmap(
    subgraph_attr, FIGURES / 'phase7_subgraph_attribution_heatmap.png')
fig_paths['hazard'] = plot_hazard_context_attribution_heatmap(
    hazard_attr, FIGURES / 'phase7_hazard_context_attribution_heatmap.png')
fig_paths['recovery'] = plot_recovery_attribution_summary(
    recovery_attr, FIGURES / 'phase7_recovery_attribution_summary.png')
fig_paths['panel'] = plot_subject_explanation_panel(
    attribution_summary, FIGURES / 'phase7_subject_explanation_panel.png')
for name, p in fig_paths.items():
    print(f'{name}: {Path(p).name}')


node: phase7_node_attribution_bar.png
subgraph: phase7_subgraph_attribution_heatmap.png
hazard: phase7_hazard_context_attribution_heatmap.png
recovery: phase7_recovery_attribution_summary.png
panel: phase7_subject_explanation_panel.png


**Figure captions (reviewer-facing):**

- *Node attribution bar* — biological domains ranked by mean contribution share to the baseline-relative graph shift.
- *Subgraph attribution heatmap* — how much each biological subgraph pattern contributes per subject-timepoint.
- *Hazard-context attribution heatmap* — HRP hazard-context alignment share (alignment, **not** exposure).
- *Recovery attribution summary* — how many metrics returned to, partially recovered toward, persisted away from, or overshot baseline.
- *Subject explanation panel* — compact table of top contributors per subject-timepoint.

## 13. Generate subject/timepoint explanation cards

Each card is a short, plain-language summary written for a non-programmer reviewer. Unavailable layers are stated explicitly.

In [12]:
import re

def _safe(s):
    return re.sub(r'[^A-Za-z0-9_.-]', '_', str(s))

card_paths = []
keys = node_attr[['subject_id', 'timepoint']].drop_duplicates()
for _, key in keys.iterrows():
    sid, tp = key['subject_id'], key['timepoint']
    s_row = attribution_summary[
        (attribution_summary['subject_id'] == sid)
        & (attribution_summary['timepoint'] == tp)]
    phase = s_row.iloc[0]['mission_phase'] if not s_row.empty else 'unknown'
    top_domain = s_row.iloc[0]['top_domain_contributor'] if not s_row.empty else 'n/a'
    top_subgraph = s_row.iloc[0]['top_subgraph_contributor'] if not s_row.empty else 'n/a'
    top_hazard = s_row.iloc[0]['top_hazard_context_contributor'] if not s_row.empty else 'n/a'
    recovery_summary = s_row.iloc[0]['recovery_summary'] if not s_row.empty else 'n/a'

    explanation = generate_subject_timepoint_explanation(
        sid, tp, node_attr, graph_attr, subgraph_attr, hazard_attr, recovery_attr)

    hazard_line = (top_hazard.replace('_', ' ') if top_hazard != 'n/a'
                   else 'hazard-context layer unavailable')
    recovery_line = (recovery_summary if recovery_summary not in ('n/a', None)
                     else 'recovery layer unavailable')

    card = (
        f'Subject: {sid}\n'
        f'Timepoint: {tp}\n'
        f'Mission phase: {phase}\n'
        f'Primary graph shift: {explanation}\n'
        f'Top biological domain contributors: {top_domain}\n'
        f'Top subgraph contributor: {top_subgraph}\n'
        f'Top HRP hazard-context alignment: {hazard_line} '
        f'(alignment, not exposure)\n'
        f'Recovery note: {recovery_line}\n'
        f'Guardrail: Monitoring-relevant pattern for expert review. '
        f'Not diagnosis, not treatment guidance, not causal proof, '
        f'not exposure measurement.\n'
    )
    cpath = CARDS / f'subject_{_safe(sid)}__timepoint_{_safe(tp)}.txt'
    cpath.write_text(card, encoding='utf-8')
    card_paths.append(cpath)

print(f'Generated {len(card_paths)} explanation cards.')
print('---- example card ----')
print(card_paths[min(2, len(card_paths) - 1)].read_text())


Generated 10 explanation cards.
---- example card ----
Subject: Demo_Crew_01
Timepoint: T2_inflight
Mission phase: inflight
Primary graph shift: For subject Demo_Crew_01 at the inflight timepoint T2_inflight, the baseline-relative graph shift is driven primarily by Body composition / physical status and Inflammation / immune-adjacent. Together these domains account for 52% of the observed node-level activation change. At the subgraph level, the largest shift maps to the cardiometabolic pattern (52% of node change). The graph-metric change is led by coactivation_edge_count (42%). The hazard-context layer aligns most strongly with Isolation and Confinement relevance, but this is not evidence of actual hazard exposure or causality. On recovery, 4 metric(s) returned near personal baseline. This attribution identifies a monitoring-relevant pattern for expert review. Guardrail: This is a monitoring-relevant pattern for expert review. It is not diagnosis, not treatment guidance, not causal pr

## 14. Generate Phase 7 report

The report aggregates input status, scope, strongest contributors per layer, recovery patterns, limitations, and the next-phase recommendation.

In [13]:
report = generate_phase7_report(
    attribution_summary, node_attr, graph_attr,
    subgraph_attr, hazard_attr, recovery_attr,
    data_provenance_note=provenance_note,
)
report_path = REPORTS / 'phase7_explainable_trajectory_attribution_report.txt'
report_path.write_text(report, encoding='utf-8')
print(f'Report written to {report_path.name}')
print(report)


Report written to phase7_explainable_trajectory_attribution_report.txt
NeuroBridge-S4 Graph Learning
Phase 7 — Explainable Within-Subject Trajectory Attribution

OVERVIEW
------------------------------------------------------------------------------
Phase 6 measured how each subject's biological adaptation graph changed from personal baseline over time. Phase 7 explains which biological domains, subgraphs, graph metrics, HRP hazard contexts, and recovery components contributed most to those baseline-relative changes. Attribution is transparent arithmetic (absolute delta / total absolute delta), not a black-box model.

INPUT DATA STATUS
------------------------------------------------------------------------------
Data provenance: schema demonstration only — not scientific evidence.
  - Node-level attribution: available
  - Graph-metric attribution: available
  - Subgraph attribution: available
  - Hazard-context attribution: available
  - Recovery attribution: available

SCOPE
--------

## 15. Conclusion

Phase 7 turns longitudinal graph deltas into explainable, reviewer-facing interpretations. It creates an evidence trail from baseline-relative graph change to biological domains, subgraphs, graph metrics, hazard-context alignment, and recovery behavior.

Every attribution number is transparent arithmetic that a reviewer can re-derive. Outputs are monitoring-relevant patterns and candidates for expert review — **not diagnosis, not treatment guidance, not causal proof, and not exposure measurement**.

**Next phase recommendation:** Phase 8 — Reference-calibrated trajectory envelope, placing each within-subject trajectory against a reference band to flag deviations that warrant closer expert review.